In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np

import utils
from utils import collect_metadata_utils, tracking_channel

utils.plotting.setup_default_plotting()

In [ ]:
# Runtime config (PROCESSING_PATH etc.) comes from `.env` -- see utils/environment_variables.py.
local_data_dir = utils.environment_variables.get_local_data_dir()
experiment_names = collect_metadata_utils.list_experiment_names(local_data_dir)
print("Available experiments:", ", ".join(experiment_names))
experiment_name = experiment_names[2]  # change index to select a different experiment
experiment_dir = local_data_dir / "collects" / experiment_name

metadata_filepath = experiment_dir / "metadata.yml"
metadata = collect_metadata_utils.load_experiment_metadata_from_file(metadata_filepath, print_summary=True)

collect_id = metadata.collect_ids[2]  # change index to select a different collect (see printed list above)

In [ ]:
# Load tracking results from file
tracking_results_directory = local_data_dir / "tracking-results"
tracking_results_directory.mkdir(parents=True, exist_ok=True)
collect_tracking_output_files = [
    fn
    for fn in tracking_results_directory.iterdir()
    if fn.name.startswith(f"{collect_id}.") and fn.name.endswith(".pkl")
]
print("Available tracking output files for this collect:")
for fn in collect_tracking_output_files:
    print(f"  {fn}")

TrackingOutputsDict = dict[str, tracking_channel.SignalTrackingOutputs]
all_tracking_loop_params: dict[str, tracking_channel.TrackingLoopParameters] = {}
all_tracking_outputs: dict[str, TrackingOutputsDict] = {}
for fn in collect_tracking_output_files:
    version_id = fn.name.split(".")[-2]
    tracking_results_filepath = os.path.join(tracking_results_directory, fn)
    with open(tracking_results_filepath, "rb") as f:
        tracking_loop_params: tracking_channel.TrackingLoopParameters = pickle.load(f)
        tracking_outputs: dict[str, tracking_channel.SignalTrackingOutputs] = pickle.load(f)
    all_tracking_loop_params[version_id] = tracking_loop_params
    all_tracking_outputs[version_id] = tracking_outputs

In [ ]:
all_tracking_version_ids = sorted(all_tracking_outputs.keys())
all_tracking_PLL_bandwidths = [
    all_tracking_loop_params[vid].PLL_bandwidth_hz for vid in all_tracking_version_ids
]
all_tracking_DLL_bandwidths = [
    all_tracking_loop_params[vid].DLL_bandwidth_hz for vid in all_tracking_version_ids
]
print("Available tracking results versions:")
for vid, pll_bw, dll_bw in zip(all_tracking_version_ids, all_tracking_PLL_bandwidths, all_tracking_DLL_bandwidths):
    print(f"  Version ID: {vid}, PLL BW: {pll_bw} Hz, DLL BW: {dll_bw} Hz")

plot_tracking_version_ids = all_tracking_version_ids
# plot_tracking_version_ids = ["v5", "v4", "v1", "v3", "v2"]

In [ ]:
# Plot prompt I/Q for each signal for a particular tracking version
plot_tracking_version_id = plot_tracking_version_ids[0]
tracking_loop_params = all_tracking_loop_params[plot_tracking_version_id]
tracking_outputs = all_tracking_outputs[plot_tracking_version_id]
plot_sig_ids = sorted(tracking_outputs.keys())
# plot_sig_ids = ["G06", "G14", "G17", "G19", "G28"]

fig = plt.figure(figsize=(12, 16), dpi=150)
title = (
    f"Tracking Results for Collect ID: {collect_id}, Version ID: {plot_tracking_version_id}\n"
    f"PLL BW: {tracking_loop_params.PLL_bandwidth_hz} Hz, DLL BW: {tracking_loop_params.DLL_bandwidth_hz} Hz"
)
utils.plotting.plot_prompt_iq_grid(fig, tracking_outputs, sig_ids=plot_sig_ids, title=title)
plt.show()

In [ ]:
# Select one sig_id and compare across tracking versions
plot_sig_id = plot_sig_ids[0]
fig = plt.figure(figsize=(12, 8), dpi=150)
utils.plotting.plot_carrier_phase_doppler_comparison(
    fig, all_tracking_outputs, all_tracking_loop_params, plot_sig_id, version_ids=plot_tracking_version_ids
)
plt.show()

In [ ]:
# Plot EPL magnitudes and code error for one tracking version
tracking_version_id = "v1"
tracking_output = all_tracking_outputs[tracking_version_id][plot_sig_id]

fig = plt.figure(figsize=(12, 8), dpi=150)
utils.plotting.plot_epl_magnitude_and_code_error(fig, tracking_output, plot_sig_id)
plt.show()